In [ ]:
# 0) Cấu hình đường dẫn dữ liệu
from pathlib import Path
DATA_PATH = Path('/mnt/data/diabetes_prediction_dataset.csv')  # thay đổi nếu cần
assert DATA_PATH.exists(), f'Không tìm thấy file: {DATA_PATH}'
DATA_PATH

In [ ]:
# 1) Load & tiền xử lý
import numpy as np
import pandas as pd

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

# Xác định cột nhãn
lower = {c.lower(): c for c in df.columns}
target_col = lower.get('outcome') or lower.get('class') or lower.get('diabetes') or lower.get('diabetic') or df.columns[-1]
y_raw = df[target_col]

# Map nhãn về 0/1 nếu là chuỗi
if y_raw.dtype == object:
    y_map = {'yes':1,'no':0,'true':1,'false':0,'positive':1,'negative':0,'pos':1,'neg':0}
    y = y_raw.astype(str).str.lower().map(y_map)
    if y.isna().any():
        y = (pd.factorize(y_raw)[0] != 0).astype(int)
else:
    y = y_raw.astype(int)

# One-hot cho cột không số (nếu có)
X = pd.get_dummies(df.drop(columns=[target_col]), drop_first=True)

X.shape, y.shape, target_col

In [ ]:
# 2) Tách tập train/val/test
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)
X_train.shape, X_val.shape, X_test.shape

In [ ]:
# 3) Khai báo mô hình + Pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

svm_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=1.0, probability=True, class_weight='balanced', max_iter=2000, random_state=42))
])

logreg_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=200, class_weight='balanced', solver='lbfgs', random_state=42))
])

svm_pipe.fit(X_train, y_train)
logreg_pipe.fit(X_train, y_train)
'done'

In [ ]:
# 4) Threshold Tuning theo F1 trên tập validation
from sklearn.metrics import precision_recall_curve
import numpy as np

def best_threshold_by_f1(y_true, proba):
    p, r, thrs = precision_recall_curve(y_true, proba)
    f1s = np.where((p+r)>0, 2*p*r/(p+r), 0)
    idx = int(np.nanargmax(f1s[1:]))  # bỏ điểm đầu tiên
    return float(thrs[idx]), float(f1s[1:][idx])

proba_val_svm = svm_pipe.predict_proba(X_val)[:,1]
proba_val_log = logreg_pipe.predict_proba(X_val)[:,1]

thr_svm, f1_svm_val = best_threshold_by_f1(y_val, proba_val_svm)
thr_log, f1_log_val = best_threshold_by_f1(y_val, proba_val_log)
thr_svm, f1_svm_val, thr_log, f1_log_val

In [ ]:
# 5) Đánh giá trên test ở ngưỡng 0.5 và ngưỡng tối ưu
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

def eval_at_threshold(y_true, proba, thr):
    y_pred = (proba >= thr).astype(int)
    return {
        'threshold': float(thr),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
    }

proba_test_svm = svm_pipe.predict_proba(X_test)[:,1]
proba_test_log = logreg_pipe.predict_proba(X_test)[:,1]

metrics = {}
for name, proba_test, thr_best in [
    ('SVM_linear', proba_test_svm, thr_svm),
    ('Logistic', proba_test_log, thr_log),
]:
    m05 = eval_at_threshold(y_test, proba_test, 0.5)
    mb = eval_at_threshold(y_test, proba_test, thr_best)
    metrics[name] = {
        'ROC_AUC': float(roc_auc_score(y_test, proba_test)),
        'PR_AUC': float(average_precision_score(y_test, proba_test)),
        'val_best_threshold': float(thr_best),
        'test@0.5': m05,
        'test@bestF1': mb
    }
metrics

In [ ]:
# 6) Vẽ ROC & Precision-Recall
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc, average_precision_score

fpr_svm, tpr_svm, _ = roc_curve(y_test, proba_test_svm)
fpr_log, tpr_log, _ = roc_curve(y_test, proba_test_log)

plt.figure()
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC={auc(fpr_svm,tpr_svm):.3f})')
plt.plot(fpr_log, tpr_log, label=f'Logistic (AUC={auc(fpr_log,tpr_log):.3f})')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC (Test)'); plt.legend();
plt.show()

prec_svm, rec_svm, _ = precision_recall_curve(y_test, proba_test_svm)
prec_log, rec_log, _ = precision_recall_curve(y_test, proba_test_log)

plt.figure()
plt.plot(rec_svm, prec_svm, label=f'SVM (AP={average_precision_score(y_test, proba_test_svm):.3f})')
plt.plot(rec_log, prec_log, label=f'Logistic (AP={average_precision_score(y_test, proba_test_log):.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision–Recall (Test)'); plt.legend();
plt.show()

In [ ]:
# 7) Lưu mô hình & summary ra đĩa (tuỳ chọn)
from joblib import dump
import json

dump(svm_pipe, '/mnt/data/svm_linear.joblib')
dump(logreg_pipe, '/mnt/data/logreg.joblib')

summary = {
    'target_col': target_col,
    'shapes': {
        'train': list(X_train.shape), 'val': list(X_val.shape), 'test': list(X_test.shape)
    },
    'metrics': metrics
}
with open('/mnt/data/eval_summary_from_notebook.json','w',encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('Saved:', '/mnt/data/svm_linear.joblib', '/mnt/data/logreg.joblib', '/mnt/data/eval_summary_from_notebook.json')